# SG13G2 Device Model Verification Framework

This notebook provides a **step-by-step walkthrough** of the complete **SG13G2 device model verification flow**, from raw measurement data parsing to simulation validation and report generation.

### 🎯 Objective

To validate that the **PDK-provided compact models** accurately reproduce **measured silicon behavior** across different corners, bias conditions, and performance metrics.

### ⚙️ Overview

The flow combines **measured data** (from `.mdm` files) with **Ngspice-based simulations** of the PDK model cards.  
It then performs automated **statistical analysis** to ensure model accuracy within defined thresholds.

### 🧩 Main Steps

1. **Environment Setup** – Define paths, dependencies, and configurations.  
2. **Data Parsing** – Extract and preprocess fab measurement data from `.mdm` files.  
3. **Simulation Execution** – Run Ngspice corner simulations (TT/FF/SS) using generated decks.  
4. **Validation & Metrics** – Compare measured vs. simulated data with tolerance rules.  
5. **Reporting** – Generate detailed summaries and markdown reports for analysis.

### 📁 Dataset

The dataset consists of **measured electrical characteristics** for various SG13G2 devices (e.g., `sg13_lv_nmos`, `sg13_lv_pmos`, etc.), stored in `.mdm` format.

Each device’s data is parsed and transformed into structured `.csv` files for subsequent validation steps.


By following this notebook, you’ll understand **how the verification framework operates**, how to **inspect validation results**, and how to **interpret model performance** across process corners.

---

## 1. Environment Setup

> Before proceeding, ensure that all **prerequisites** are satisfied as described in the [Prerequisites section](../README.md#prerequisites).

This step prepares your environment to execute the SG13G2 model validation flow.

---

### 🧱 Prerequisites Checklist

Before running the notebook, make sure you have:

- ✅ **Built the OSDI models**  
  Refer to the [Verilog-A README](../../../../verilog-a/README.md) for instructions on compiling Verilog-A models to OSDI format.

- ✅ **Created and activated a Python virtual environment**  
  Use your preferred tool (`venv`, `conda`, or `poetry`) to isolate dependencies.

- ✅ **Installed all Python requirements**

---

### ⚙️ Once Complete

After completing the above setup steps:

- You’ll be able to **import all required modules** — including the MDM parser, ngspice simulation interface, and model validation engine.  
- You can **run the notebook end-to-end**, from data parsing and simulation to statistical verification and report generation.

## 2. Device Data Exploration and Validation Overview

In this section, we’ll explore how the verification system operates — from running device-level tests to inspecting the validation results.

You’ll:

- 🧩 **Run device verification tests** interactively (using configuration files).  
- 📊 **Visualize measured vs. simulated data**, comparing results across TT, FF, and SS corners.  
- 🧠 **Analyze out-of-bound (OOB) points** to understand model accuracy and deviation sources.  
- 🧾 **Review test outcomes and assertion reports**, including pass/fail summaries and diagnostic logs.

This part serves as both an *exploration tool* and a *debugging aid* — helping you quickly identify where the simulated models diverge from silicon measurements, and whether they remain within the expected process envelope.

In [ ]:
# ==============================================================
# Environment Initialization
# ==============================================================
# This cell prepares the runtime environment for the model
# verification and data exploration workflow.
# ==============================================================

# --- Standard Library Imports ---
import os
import sys
import logging
import yaml
import tempfile
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

# --- Data and Plotting Libraries ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# --- Interactive Visualization (Jupyter Widgets) ---
from ipywidgets import (
    VBox, HBox, Dropdown, Button, Output, Label, HTML, Layout, Accordion, IntProgress, BoundedIntText
)
from IPython.display import display, clear_output, Markdown

# ==============================================================
# Path Setup
# ==============================================================
# Move one directory up from 'workflow_notebooks/' → 'devices/',
# then add it to Python’s search path to enable local imports.
# ==============================================================

TESTS_DIR = Path.cwd().parent
sys.path.append(str(TESTS_DIR))

# ==============================================================
# Project-Specific Imports
# ==============================================================
# Import test case configurations and verification core logic.
# ==============================================================

from models_verifier.constants import CASES
from models_verifier.models_verifier import MdmVerifier

# --- Status ---
print("✅ Environment initialized successfully.")
print(f"📂 Test directory set to: {TESTS_DIR}")

In [ ]:
# ==============================================================
# Load Available Device Test Cases
# ==============================================================
# This cell initializes the mapping of all supported device test cases 
# defined in `models_verifier.constants.CASES`.
#
# Each entry maps a device label (e.g., "pmos_lv") to its corresponding 
# configuration file, which contains all setup parameters for that device.
#
# Example entry:
#   pmos_lv → configs/mos/pmos_lv/sg13_lv_pmos.yaml
# ==============================================================

CASE_MAP = {label: rel for label, rel in CASES}

print("\n📘 Available Device Test Cases")
print("=" * 60)
print(f"{'Device Label':<20} | {'Configuration File (Relative Path)'}")
print("-" * 60)
for label, rel_path in CASE_MAP.items():
    print(f"{label:<20} | {rel_path}")
print("=" * 60)
print(f"✅ Total: {len(CASE_MAP)} device configurations available.\n")

In [ ]:
# ==============================================================
# Update Relative Paths in Config
# ==============================================================
# Some configuration YAMLs use paths relative to the test directory.
# This helper reads the original YAML, converts relative paths to
# absolute, writes a temporary YAML, and returns its path.
# ==============================================================

def update_relative_paths(cfg_path: Path, base_dir: Path) -> Path:
    """
    Read a YAML config from cfg_path, convert relative paths to absolute
    (based on base_dir), then write the updated config to a temporary
    YAML file and return its path.
    Only updates keys that exist: mdm_dir, corner_lib_path, osdi_path.
    """
    with open(cfg_path) as f:
        config = yaml.safe_load(f)

    path_keys = ["mdm_dir", "corner_lib_path", "osdi_path", "dc_template_path", "output_dir"]
    for key in path_keys:
        if key in config and config[key]:
            p = Path(config[key])
            if not p.is_absolute():
                config[key] = str((base_dir / p).resolve())

    # Write updated config to a temporary YAML file
    tmp_file = tempfile.NamedTemporaryFile(mode="w", delete=False, suffix=".yaml")
    yaml.safe_dump(config, tmp_file)
    tmp_file.close()
    return Path(tmp_file.name)

In [ ]:
# ==============================================================
# Run Device with Live Progress
# ==============================================================

def run_one_device(label: str, progress_bar: IntProgress):
    """
    Run the device verification workflow for a given device label.
    Updates a live progress bar incrementally.
    """
    global verifier
    verifier = None

    cfg_org = TESTS_DIR / CASE_MAP[label]
    cfg_path = None

    try:
        cfg_path = update_relative_paths(cfg_org, TESTS_DIR)
        if not cfg_path.exists():
            print(f"❌ Config YAML not found: {cfg_path}")
            return

        verifier = MdmVerifier(cfg_path)
        print(f"🚀 Starting verification of device: {label}")

        # --- Setup live progress bar ---
        progress_bar.value = 0
        progress_bar.max = 100
        progress_bar.description = "Running..."
        progress_bar.bar_style = "info"

        # Attach progress callback if verifier supports it
        if hasattr(verifier, "progress_callback"):
            def _update_progress(p):
                # p is expected 0.0 → 1.0
                progress_bar.value = int(p * 100)
            verifier.progress_callback = _update_progress

        # Run the verification
        verifier.run_verification()

        # Complete progress
        progress_bar.value = 100
        progress_bar.description = "Done ✅"
        progress_bar.bar_style = "success"

    except Exception as e:
        progress_bar.description = "Failed ❌"
        progress_bar.bar_style = "danger"
        print(f"❌ Verification failed: {e}")

    finally:
        # --- Cleanup temp config ---
        if cfg_path and cfg_path.exists():
            try:
                os.remove(cfg_path)
                print(f"🗑 Cleaned up temporary config file: {cfg_path.name}")
            except Exception as e:
                print(f"⚠️ Failed to cleanup temp config {cfg_path.name}: {e}")

In [ ]:
# ==============================================================
# Interactive Device Runner UI (with isolated summary)
# ==============================================================

# --- Widgets ---
device_dropdown = Dropdown(
    options=list(CASE_MAP.keys()),
    description="Device:",
    layout=Layout(width="350px")
)
run_btn = Button(
    description="Run Test", button_style="success", icon="play-circle",
    layout=Layout(width="150px")
)
progress = IntProgress(
    value=0, min=0, max=100, description="Ready to Run", bar_style="info",
    layout=Layout(width="350px")
)

def _run_clicked(_):
    """
    Callback: only updates progress bar via run_device.
    Does NOT print the summary here.
    """
    try:
        run_one_device(device_dropdown.value, progress)
    except Exception as e:
        progress.bar_style = "danger"
        progress.description = "Failed ❌"
        print(f"[ERROR] {e}")

# Clear old callbacks and bind
run_btn._click_handlers.callbacks.clear()
run_btn.on_click(_run_clicked)

# Display UI
ui_box = VBox([
    HTML("<h3>🧪 Device Verification Runner</h3>"),
    HBox([device_dropdown, run_btn]),
    progress
])
display(ui_box)

In [ ]:
def print_final_summary():
    """
    Print the final_summary.md of the last run as Markdown.
    """
    global verifier
    if verifier is None:
        print("No device run yet.")
        return

    final_summary_path = verifier.output_dir / "final_reports" / "final_summary.md"
    if not final_summary_path.exists():
        print(f"No final_summary.md found at: {final_summary_path}")
        return

    # Read the content
    with open(final_summary_path, "r") as f:
        content = f.read()

    # Display as Markdown
    display(Markdown(content))

# Call the function to show the final summary
print_final_summary()

In [110]:
# ==============================================================
# Load and Merge Simulation Results
# ==============================================================
# This utility function loads all combined NGSPICE simulation results
# for a given device test (via the `MdmVerifier` instance).
#
# Each result file (CSV) under:
#     <output_dir>/combined_results/
# is read into a Pandas DataFrame and keyed by its simulation setup type.
#
# Notes:
#   - If `master_setup_type` column is missing, the filename stem is used.
#   - Adds a column `source_csv` for traceability.
# ==============================================================

def load_simulation_results(verifier: MdmVerifier) -> Dict[str, pd.DataFrame]:
    """
    Load merged simulation CSVs for all corner setups of a given device.

    Parameters
    ----------
    verifier : MdmVerifier
        Active verification object containing the `output_dir` path.

    Returns
    -------
    Dict[str, pd.DataFrame]
        Dictionary mapping each setup type (e.g., "ff", "ss", "tt") to
        its corresponding DataFrame.
    """
    sim_dir = (verifier.output_dir / "combined_results").resolve()
    results: Dict[str, pd.DataFrame] = {}

    if not sim_dir.exists():
        print(f"⚠️ No simulation directory found at: {sim_dir}")
        return results

    csv_files = sorted(sim_dir.glob("*.csv"))
    if not csv_files:
        print(f"⚠️ No CSV files found in: {sim_dir}")
        return results

    print(f"📂 Loading simulation results from: {sim_dir}")
    print("-" * 60)

    for csv_path in csv_files:
        df = pd.read_csv(csv_path)
        setup_key = (
            str(df.get("master_setup_type", pd.Series([csv_path.stem])).iloc[0])
            if not df.empty else csv_path.stem
        )
        df["source_csv"] = csv_path.name
        results[setup_key] = df
        print(f"✅ Loaded: {csv_path.name:<25} → setup: {setup_key}")

    print("-" * 60)
    print(f"📊 Total setups loaded: {len(results)}")
    return results

# ==============================================================
# Load Simulation Results for the Verified Device
# ==============================================================
# Once the verification run is complete, this cell loads all
# combined NGSPICE simulation data (TT, FF, SS, etc.) for
# visualization and post-analysis.
# ==============================================================

SIM_MERGED_BY_SETUP = load_simulation_results(verifier)

📂 Loading simulation results from: /home/farag/Team_mabrains/ihp/ihp_os/IHP-Open-PDK/ihp-sg13g2/libs.tech/ngspice/testing/devices/models_results/nmos_lv/combined_results
------------------------------------------------------------
✅ Loaded: dc_idvd.csv               → setup: dc_idvd
✅ Loaded: dc_idvg.csv               → setup: dc_idvg
✅ Loaded: di_bd_area.csv            → setup: di_bd_area
✅ Loaded: di_bd_perim.csv           → setup: di_bd_perim
✅ Loaded: di_bs_area.csv            → setup: di_bs_area
✅ Loaded: di_bs_perim.csv           → setup: di_bs_perim
------------------------------------------------------------
📊 Total setups loaded: 6


In [ ]:
# ==============================================================
# Simulation Data Utilities: Parsing, Masking, and Plotting
# ==============================================================
# This cell contains helper functions for:
#   - Styling matplotlib axes
#   - Extracting simulation columns
#   - Identifying out-of-bound points
#   - Formatting panel titles and device parameters
# ==============================================================

def style_axes(ax: plt.Axes, sweep_var: str, output_var: str) -> None:
    """
    Standardize axes styling for all device plots.
    
    Parameters
    ----------
    ax : plt.Axes
        Matplotlib Axes object to style.
    sweep_var : str
        Name of the sweep variable (X-axis label).
    output_var : str
        Name of the output variable (Y-axis label).
    """
    ax.set_xlabel(str(sweep_var))
    ax.set_ylabel(str(output_var))
    ax.grid(True, alpha=0.25)
    sf = ScalarFormatter(useMathText=True)
    sf.set_powerlimits((-3, 3))
    ax.yaxis.set_major_formatter(sf)


def split_sim_columns(df: pd.DataFrame, out_first: str) -> Tuple[List[str], Optional[str], List[str]]:
    """
    Identify simulation columns corresponding to a particular output variable.

    Parameters
    ----------
    df : pd.DataFrame
        Block of simulation results.
    out_first : str
        Base output variable name (prefix for simulated columns).

    Returns
    -------
    sim_cols : List[str]
        All columns starting with `{out_first}_sim_`.
    typ_col : Optional[str]
        Column corresponding to the typical/TT simulation (if present).
    other_cols : List[str]
        Remaining simulation columns excluding the typical column.
    """
    prefix = f"{out_first}_sim_"
    sim_cols = [c for c in df.columns if c.startswith(prefix)]
    if not sim_cols:
        return [], None, []

    typ_candidates = [c for c in sim_cols if c.split("_")[-1].lower() in {"typ", "tt", "typical"}]
    typ_col = typ_candidates[0] if typ_candidates else None
    others = [c for c in sim_cols if c != typ_col]
    return sim_cols, typ_col, others


def outside_mask(y: np.ndarray, lo: np.ndarray, hi: np.ndarray) -> np.ndarray:
    """
    Return a boolean mask for points outside the given lower/upper bounds.

    Parameters
    ----------
    y : np.ndarray
        Values to check.
    lo : np.ndarray
        Lower bound.
    hi : np.ndarray
        Upper bound.

    Returns
    -------
    mask : np.ndarray
        Boolean array where True indicates out-of-bound points.
    """
    return (y < lo) | (y > hi)


def parse_block_outputs(block: pd.DataFrame) -> List[str]:
    """
    Parse the 'output_vars' column of a DataFrame block into a list of output names.

    Parameters
    ----------
    block : pd.DataFrame
        DataFrame with a column 'output_vars' (comma-separated string).

    Returns
    -------
    outputs : List[str]
        List of cleaned output variable names.
    """
    if "output_vars" not in block.columns or pd.isna(block["output_vars"].iloc[0]):
        return []
    return [s.strip() for s in str(block["output_vars"].iloc[0]).split(",") if s.strip()]


def device_param_columns(device_type_hint: str) -> list[str]:
    """
    Map device type hints to relevant parameter columns for display.

    Parameters
    ----------
    device_type_hint : str
        Hint about the device type (e.g., 'mos', 'hbt').

    Returns
    -------
    List[str]
        Columns to include in panel titles.
    """
    mapping = {
        'mos':    ['w','l','ad','as','pd','ps','nf','m'],
        'pnpmpa': ['a','p'],
        'hbt':    ['ve','vs','l','w','m','nx'],
    }
    dt = (device_type_hint or "").lower()
    if dt in mapping:
        return mapping[dt]
    for k, v in mapping.items():
        if k in dt:
            return v
    return []


def format_panel_title(block: pd.DataFrame, out_name: str, device_type_hint: str) -> str:
    """
    Generate a descriptive panel title for plotting, including device parameters,
    temperature, and source data.

    Parameters
    ----------
    block : pd.DataFrame
        DataFrame row corresponding to the plotted block.
    out_name : str
        Name of the output variable.
    device_type_hint : str
        Hint about device type to select parameter columns.

    Returns
    -------
    title : str
        Formatted panel title for display above the plot.
    """
    cols = device_param_columns(device_type_hint)
    row = block.iloc[0]  # Take the first row as representative
    parts = []

    # Device parameter values
    for c in cols:
        if c in block.columns and pd.notna(row.get(c, np.nan)):
            val = row[c]
            if isinstance(val, float):
                val = f"{val:.4g}"
            parts.append(f"{c}={val}")

    # Temperature string
    t_str = f"T={row['temp']}°C" if "temp" in block.columns and pd.notna(row.get("temp", np.nan)) else None

    # Source/input data string
    src_str = str(row["input_data"]) if "input_data" in block.columns and pd.notna(row.get("input_data", np.nan)) else None

    # Left side: output variable + source
    left = " — ".join(filter(None, [out_name.upper(), f"[{src_str}]" if src_str else None]))

    # Right side: temperature and parameter details
    right_bits = [t_str] if t_str else []
    if parts:
        right_bits.append(", ".join(parts))
    right = " — " + " | ".join(right_bits) if right_bits else ""

    return left + right

In [ ]:
# ==============================================================
# Interactive Plotting: Compare Measured vs Simulation Blocks
# ==============================================================
# This function generates a grid of panels for each (block_id, output_var),
# plotting measured data, simulation typical curves, and min/max ranges.
# ==============================================================

def plot_blocks_comparison(
    df: pd.DataFrame,
    device_type_hint: str = "mos",
    block_ids: Optional[List[str]] = None,
    figsize: Tuple[float, float] = (10, 6),
    ncols: int = 3,
    max_blocks: Optional[int] = 20,
) -> Tuple[plt.Figure, List[plt.Axes]]:
    """
    Create a grid of panels where each (block_id, output_var) gets its own subplot.

    Parameters
    ----------
    df : pd.DataFrame
        Wide-format DataFrame with columns:
          - block_id
          - sweep_var
          - output_vars (comma-separated)
          - temp, w, l (optional)
          - measured and simulation columns (e.g., y_meas, y_sim_tt)
    device_type_hint : str
        Device type hint (e.g., "mos", "hbt") for title formatting.
    block_ids : Optional[List[str]]
        Specific block IDs to plot; defaults to all.
    figsize : Tuple[float, float]
        Base figure size for a single subplot.
    ncols : int
        Number of columns in subplot grid.
    max_blocks : Optional[int]
        Maximum number of blocks to plot.

    Returns
    -------
    fig : plt.Figure
        Matplotlib figure object.
    axes : List[plt.Axes]
        Flattened list of axes corresponding to each panel.
    """
    # Clear previous figure outputs in interactive notebook
    plt.close('all')

    COLORS = {
        "range_fill": "#ccc2c2",
        "typ":        "#2F2F2F",
        "sim_alt":    "#9A9A9A",
        "meas":       "#1f77b4",
        "outside":    "#d62728",
    }

    if block_ids is None:
        block_ids = list(pd.unique(df["block_id"]))
    if max_blocks is not None:
        block_ids = block_ids[:max_blocks]

    # Build panel list: all outputs for each block
    panels: List[Tuple[str, str]] = []
    for bid in block_ids:
        block = df[df["block_id"] == bid]
        if block.empty:
            continue
        for out_name in parse_block_outputs(block):
            panels.append((bid, out_name))

    # Handle empty panels
    if not panels:
        fig, ax = plt.subplots(1, 1, figsize=figsize)
        ax.text(0.5, 0.5, "No panels to plot.", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
        plt.tight_layout()
        return fig, [ax]

    # Create grid
    nrows = int(np.ceil(len(panels) / ncols))
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(figsize[0] * ncols, figsize[1] * nrows),
        squeeze=False
    )
    axes_flat = axes.flatten()

    for i, (bid, out_name) in enumerate(panels):
        ax = axes_flat[i]
        block = df[df["block_id"] == bid].copy()
        if block.empty:
            ax.text(0.5, 0.5, f"No data for\n{bid}", ha="center", va="center", transform=ax.transAxes)
            ax.set_axis_off()
            continue

        sweep_var = str(block["sweep_var"].iloc[0])
        block = block.sort_values(sweep_var)
        x = block[sweep_var].to_numpy()

        meas_col = f"{out_name}_meas"
        sim_cols, typ_col, other_cols = split_sim_columns(block, out_name)

        # Envelope from non-typical corners
        y_lo = y_hi = None
        if other_cols:
            sims = block[other_cols].to_numpy(dtype=float)
            y_lo = np.min(sims, axis=1)
            y_hi = np.max(sims, axis=1)
            ax.fill_between(x, y_lo, y_hi, alpha=0.75, color=COLORS["range_fill"], label="Simulation Range")

        # Typical baseline or all simulations if no typ
        if typ_col is not None:
            ax.plot(
                x, block[typ_col].to_numpy(dtype=float),
                "--", linewidth=1.8, color=COLORS["typ"],
                label=typ_col.replace(f"{out_name}_sim_", "").upper()
            )
        else:
            for c in sim_cols:
                ax.plot(
                    x, block[c].to_numpy(dtype=float),
                    "--", linewidth=1.2, alpha=0.7, color=COLORS["sim_alt"],
                    label=c.replace(f"{out_name}_sim_", "").upper()
                )

        # Measured
        if meas_col in block.columns:
            y_meas = block[meas_col].to_numpy(dtype=float)
            ax.plot(x, y_meas, "-o", linewidth=1.8, markersize=3.5, color=COLORS["meas"], label="Measured")
            if y_lo is not None and y_hi is not None:
                mask = outside_mask(y_meas, y_lo, y_hi)
                if np.any(mask):
                    ax.scatter(
                        x[mask], y_meas[mask],
                        marker="x", s=64, linewidths=2.2,
                        color=COLORS["outside"], label="Measured Outside Range", zorder=3
                    )

        # Title and axes styling
        ax.set_title(format_panel_title(block, out_name, device_type_hint), fontsize=10)
        style_axes(ax, sweep_var, out_name)

        # De-duplicate legend entries
        handles, labels = ax.get_legend_handles_labels()
        seen, H, L = set(), [], []
        for h, lab in zip(handles, labels):
            if lab not in seen:
                seen.add(lab)
                H.append(h)
                L.append(lab)
        if H:
            ax.legend(H, L, fontsize=8, frameon=False, loc="best")

    # Hide unused axes
    for j in range(len(panels), len(axes_flat)):
        axes_flat[j].set_visible(False)

    plt.tight_layout()
    return fig, list(axes_flat[:len(panels)])

In [ ]:
# ==============================================================
# Plot a Single CSV Type (Simulation Setup) for Multiple Panels
# ==============================================================
# This function renders up to `panels` (block_id, output_var) plots
# from a single simulation type (e.g., "tt", "ss", "ff").
# ==============================================================

def plot_csv_type(
    setup_type: str,
    panels: int = 20,
    ncols: int = 4,
    show: bool = True
) -> Optional[Tuple[plt.Figure, List[plt.Axes]]]:
    """
    Plot multiple (block_id, output) panels from a single simulation setup.

    Parameters
    ----------
    setup_type : str
        Simulation setup key, e.g., "tt", "ff", "ss".
    panels : int
        Maximum number of blocks to plot.
    ncols : int
        Number of columns in the subplot grid.
    show : bool
        Whether to display the figure in the notebook (default: True).

    Returns
    -------
    fig, axes : tuple of (plt.Figure, List[plt.Axes])
        Matplotlib figure and flattened axes list for each panel.
        Returns None if the setup_type is not loaded or empty.
    """
    # Verify global simulation results exist
    if "SIM_MERGED_BY_SETUP" not in globals() or setup_type not in SIM_MERGED_BY_SETUP:
        print(f"[ERROR] Simulation type '{setup_type}' not loaded. Run a device first.")
        return None

    df = SIM_MERGED_BY_SETUP[setup_type]
    if df.empty:
        print(f"[WARN] Simulation DataFrame for '{setup_type}' is empty.")
        return None

    # Select up to `panels` unique block_ids
    block_ids = list(pd.unique(df["block_id"]))[:panels]
    if not block_ids:
        print(f"[WARN] No block_id found in '{setup_type}'.")
        return None

    # Generate the comparison plots
    fig, axes = plot_blocks_comparison(
        df=df,
        block_ids=block_ids,
        ncols=ncols,
        max_blocks=panels,
        device_type_hint=getattr(verifier, "device_type", "mos")
    )

    # Ensure axes is iterable
    if not isinstance(axes, (list, np.ndarray)):
        axes = [axes]

    # Apply grid and log scale
    for ax in axes:
        if ax is not None:
            ax.grid(True, which="both", linestyle="--", linewidth=0.7, alpha=0.7)
            ax.set_yscale("symlog", linthresh=1e-15)

    if show:
        display(fig)

    return fig, axes

In [ ]:
# ==============================================================
# Batch Plot: Render All Simulation Types
# ==============================================================

PANELS_PER_TYPE = 5  # Number of panels per type
NCOLS           = 1  # Number of columns in the grid

# --- Check for simulation results ---
if "SIM_MERGED_BY_SETUP" not in globals() or not SIM_MERGED_BY_SETUP:
    print("[ERROR] No simulation data loaded. Run a device first.")
else:
    sim_types = sorted(SIM_MERGED_BY_SETUP.keys())
    
    for setup_type in sim_types:
        print(f"\n📂 Simulation Type: {setup_type}")
        fig_axes = plot_csv_type(
            setup_type=setup_type,
            panels=PANELS_PER_TYPE,
            ncols=NCOLS,
            show=True
        )
        
        if fig_axes is None:
            print(f"[WARN] No panels plotted for '{setup_type}'.")
        else:
            fig, axes = fig_axes
            # Optional: could save figures here if needed
            # fig.savefig(f"{setup_type}_comparison.png", dpi=150)

In [ ]:
# ==============================================================
# Interactive Enhanced Plot Widget for Simulation Data
# ==============================================================

class EnhancedPlotWidget:
    """
    Interactive widget for visualizing SG13G2 simulation results.
    Allows selection of simulation setup, filtering by device parameters,
    and dynamic plotting of multiple blocks and outputs.
    """
    def __init__(self):
        # Available simulation types
        self.types_available = sorted(SIM_MERGED_BY_SETUP.keys()) if "SIM_MERGED_BY_SETUP" in globals() else []

        # --- UI Controls ---
        self.type_dd = Dropdown(options=self.types_available, description="Setup:")
        self.count_in = BoundedIntText(value=20, min=1, max=500, description="Max Panels:")
        self.cols_in = BoundedIntText(value=1, min=1, max=8, description="Grid Cols:")
        self.plot_btn = Button(description="Generate Plot", button_style="primary")
        self.output = Output()
        self.param_filter_container = VBox()

        # Mapping of param name → dropdown widget
        self.param_filters = {}

        # --- Event bindings ---
        self.type_dd.observe(self._on_type_change, names='value')
        self.plot_btn.on_click(self._generate_plot)

        # Initialize filters if any types available
        if self.types_available:
            self.type_dd.value = self.types_available[0]
            self._create_filters()

    # ---------------------------
    # Helper Methods
    # ---------------------------
    def _format_dropdown_options(self, values):
        """Formats numeric values for dropdown display."""
        formatted_options = [(str(float(f"{float(val):.12g}")), val) for val in sorted(values)]
        return formatted_options

    def _get_device_param_columns(self, device_type_hint):
        """Returns relevant device parameters for filters."""
        device_param_map = {
            'mos': ['temp', 'w', 'l', 'nf', 'm'],
            'pnpmpa': ['a'],
            'hbt': ['ve', 'vs', 'temp', 'l', 'w', 'm', 'nx'],
        }
        dt_lower = device_type_hint.lower()
        if dt_lower in device_param_map:
            return device_param_map[dt_lower]
        for key, val in device_param_map.items():
            if key in dt_lower:
                return val
        return []

    def _on_type_change(self, change):
        """Rebuild filter widgets when the selected simulation type changes."""
        self._create_filters()

    def _create_filters(self):
        """Generate dropdowns for device parameter filters."""
        if not self.type_dd.value or self.type_dd.value not in SIM_MERGED_BY_SETUP:
            return
        df = SIM_MERGED_BY_SETUP[self.type_dd.value]
        if df.empty: return

        self.param_filters.clear()
        param_cols = self._get_device_param_columns(getattr(verifier, "device_type", "mos"))
        param_widgets = []

        for col in param_cols:
            if col in df.columns and df[col].nunique() > 1:
                unique_vals = df[col].dropna().unique()
                options = [('All', 'All')] + self._format_dropdown_options(unique_vals)
                widget = Dropdown(options=options, value='All', description=f"{col}:", layout={'width': '200px'})
                self.param_filters[col] = widget
                param_widgets.append(widget)

        # Arrange in rows of 4
        self.param_filter_container.children = [HTML("<b>Device Parameters:</b>")] + \
            [HBox(param_widgets[i:i+4]) for i in range(0, len(param_widgets), 4)]

    def _get_filtered_data(self):
        """Apply all selected filters to the dataframe."""
        if not self.type_dd.value or self.type_dd.value not in SIM_MERGED_BY_SETUP:
            return pd.DataFrame()

        df = SIM_MERGED_BY_SETUP[self.type_dd.value].copy()
        for col, widget in self.param_filters.items():
            if hasattr(widget, "value") and widget.value != 'All' and col in df.columns:
                val = widget.value
                if pd.api.types.is_numeric_dtype(df[col]):
                    df = df[np.isclose(df[col], val)]
                else:
                    df = df[df[col] == val]
        return df

    def _generate_plot(self, button=None):
        """Main plotting function triggered by 'Generate Plot' button."""
        plt.close('all')
        with self.output:
            self.output.clear_output()
            filtered_df = self._get_filtered_data()

            if filtered_df.empty:
                print("[INFO] No data remains after applying selected filters.")
                return

            print(f"Original rows: {len(SIM_MERGED_BY_SETUP[self.type_dd.value])}")
            print(f"Filtered rows: {len(filtered_df)}")

            block_ids = list(filtered_df["block_id"].unique())
            max_panels = min(len(block_ids), self.count_in.value)
            print(f"Plotting {max_panels} panels from {len(block_ids)} unique blocks...")

            try:
                fig, axes = plot_blocks_comparison(
                    df=filtered_df,
                    device_type_hint=getattr(verifier, "device_type", "mos"),
                    block_ids=block_ids[:max_panels],
                    ncols=self.cols_in.value,
                    max_blocks=max_panels
                )

                # Ensure axes is iterable
                if not isinstance(axes, (list, np.ndarray)):
                    axes = [axes]

                # Apply grid and log scale
                for ax in axes:
                    if ax is not None:
                        ax.grid(True, which="both", linestyle="--", linewidth=0.7, alpha=0.7)
                        ax.set_yscale("symlog", linthresh=1e-15)

                display(fig)
                plt.close(fig)
                print("✅ Plot generated successfully!")
            except Exception as e:
                print(f"[ERROR] Failed to generate plot: {e}")

    # ---------------------------
    # Display Method
    # ---------------------------
    def display(self):
        """Render the full widget UI in the notebook."""
        main_controls = VBox([
            Label(f"Select Setup and plot parameters for {getattr(verifier, 'device_type', 'device')}"),
            HBox([self.type_dd, self.count_in, self.cols_in])
        ])
        interface = VBox([
            HTML("<p>Select a Setup, apply filters using dropdowns, then click 'Generate Plot'.</p>"),
            main_controls,
            self.param_filter_container,
            HBox([self.plot_btn]),
            self.output
        ])
        return interface

# Instantiate and render the widget
enhanced_widget = EnhancedPlotWidget()
display(enhanced_widget.display())